# C4-classical-ml-practice — Practice p21 — Solution

The interaction is a deterministic rowwise calculation, so it does not consult labels or fitted statistics. Keeping it in the pipeline nevertheless gives every training, validation, and later prediction row the same path; the split happens before `pipe.fit`, and the scaler and kNN therefore see only the 170 training rows during fitting.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

SEED = 20260804
rng = np.random.default_rng(SEED)
n_rows = 233
X = pd.DataFrame({
    "ridge_temp_c": rng.normal(14.3, 3.7, n_rows),
    "soil_moisture_pct": rng.normal(46.1, 8.3, n_rows),
    "slope_deg": rng.normal(17.9, 4.1, n_rows),
})
latent = X["ridge_temp_c"] * X["soil_moisture_pct"] - 9.7 * X["slope_deg"]
y = (latent + rng.normal(0.0, 41.0, n_rows) > np.median(latent)).astype(int).to_numpy()
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.27, random_state=SEED, stratify=y
)


def add_interaction(frame):
    derived = frame.copy()
    derived["temp_x_moisture"] = (
        derived["ridge_temp_c"] * derived["soil_moisture_pct"]
    )
    return derived


pipe = Pipeline([
    ("derive", FunctionTransformer(add_interaction, validate=False)),
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7)),
])
pipe.fit(X_tr, y_tr)
derived_probe = pipe.named_steps["derive"].transform(X_tr.iloc[:2])
n_features_after = int(derived_probe.shape[1])
val_acc = float(pipe.score(X_val, y_val))

### Answer check

In [ ]:
assert list(pipe.named_steps) == ["derive", "scale", "knn"]
assert pipe.named_steps["knn"].n_neighbors == 7
assert "temp_x_moisture" not in X.columns
assert "temp_x_moisture" not in X_tr.columns
assert "temp_x_moisture" not in X_val.columns
assert n_features_after == 4
assert np.allclose(
    derived_probe["temp_x_moisture"].to_numpy(),
    (
        X_tr.iloc[:2]["ridge_temp_c"]
        * X_tr.iloc[:2]["soil_moisture_pct"]
    ).to_numpy(),
    atol=1e-12,
    rtol=0,
)
assert np.isclose(val_acc, 58 / 63, atol=1e-12, rtol=0)